# Capítulo 31 · Laboratório 31: juntando tudo, o Método 6C testado
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026** (40 ações desde 2005, Ibovespa e S&P 500 desde 2004). Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap31.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. As 40 ações passam pela limpeza do Capítulo 28. Custo de 0,10% por operação. O sorteio da Parte 3 usa semente fixa. A Parte 3 leva cerca de um minuto. O painel da Parte 5 é um exemplo de rotina em 30/09/2026: **não é recomendação**.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import re
import time
import numpy as np
import pandas as pd
import yfinance as yf

B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
FIM, CUSTO, PRAZO = "2026-10-01", 0.001, 20
CORTE = "2016-01-01"                      # dentro da amostra: 2006 a 2015; fora: 2016 a 2026
rng = np.random.default_rng(33)

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
CACHE = "dados_cap31.pkl"


def mostra(*partes):
    """print com vírgula decimal e ponto de milhar (padrão brasileiro)."""
    texto = " ".join(str(p) for p in partes)
    texto = re.sub(r"(?<=\d)\.(?=\d)", ",", texto)
    print(re.sub(r"(?<=\d)_(?=\d)", ".", texto))


def baixa(*args, **kw):
    for tentativa in range(5):                 # o provedor às vezes falha: tenta de novo
        try:
            x = yf.download(*args, progress=False, **kw)
            if len(x):
                return x
        except Exception:
            pass
        time.sleep(5 * (tentativa + 1))
    ativo = args[0] if isinstance(args[0], str) else f"a lista de {len(args[0])} ativos"
    raise RuntimeError(f"Não consegui baixar {ativo} do Yahoo Finance depois de 5 tentativas. O provedor pode estar"
                       " fora do ar ou limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):                  # cópia local: o provedor pode corrigir o histórico entre execuções
    dados, ibov, spx = pd.read_pickle(CACHE)
else:
    dados = baixa(TICKERS, start="2005-01-01", end=FIM, auto_adjust=True, group_by="ticker")
    ibov = baixa("^BVSP", start="2004-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    spx = baixa("^GSPC", start="2004-01-01", end=FIM, auto_adjust=True, multi_level_index=False)
    pd.to_pickle((dados, ibov, spx), CACHE)


CORRIGIDAS = {}                            # o que a limpeza mudou (registro da Parte 5; sem repetir)
SUSPEITOS = {"EQTL3.SA": ("2010-08-25", "queda de 40% sem volta (provável ajuste de bonificação ou desdobramento faltando)")}


def limpa(d, nome=""):
    """Limpeza do Capítulo 26, mais três regras para dado sujo que engana o teste de stops:
    1) máxima impossível: pavio de mais de 50% acima da abertura e do fechamento vira o maior dos dois;
    2) desdobramento ou grupamento sem ajuste: salto de mais de 2 vezes (ou menos da metade) da abertura em relação ao
       fechamento anterior, sem volta nos 5 pregões seguintes, corrige o histórico anterior pela mesma proporção;
    3) buraco nos dados: com mais de 15 dias corridos sem cotação, só fica o trecho contínuo depois do último buraco."""
    d = d.dropna().copy()
    ok = (d[["Open", "High", "Low", "Close"]] > 0).all(axis=1) & (d["High"] > d["Low"])
    d = d[ok & (d["Volume"] > 0)] if "Volume" in d else d[ok]
    px = ["Open", "High", "Low", "Close"]
    topo = d[["Open", "Close"]].max(axis=1)
    alta = d["High"] > 1.5 * topo
    for dt in d.index[alta]:
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: máxima {d.at[dt, 'High']:.2f} com fechamento {d.at[dt, 'Close']:.2f}", None)
    d.loc[alta, "High"] = topo[alta]
    salto = d["Open"] / d["Close"].shift()
    dura = d["Close"].shift(-5) / d["Close"].shift()
    for dt in d.index[((salto < 0.5) | (salto > 2)) & ((dura < 0.6) | (dura > 1.6))]:
        f_ = salto.loc[dt]
        d.loc[d.index < dt, px] *= f_
        CORRIGIDAS.setdefault(f"{nome} {dt:%d/%m/%Y}: salto de {f_:.4f} vezes sem volta (ajuste que faltou); histórico"
                          " anterior corrigido", None)
    if nome == "SUZB3.SA":                 # antes disso, preço parado no Yahoo (Capítulo 26)
        d = d.loc["2017-11-10":]
    if nome in SUSPEITOS:                  # queda sem volta que parece ajuste faltando, mas sem prova: fica fora
        dt, motivo = SUSPEITOS[nome]
        CORRIGIDAS.setdefault(f"{nome} {pd.Timestamp(dt):%d/%m/%Y}: {motivo}; usa só a partir dessa data", None)
        d = d.loc[dt:]
    buraco = d.index.to_series().diff().dt.days
    if (buraco > 15).any():
        ini = buraco[buraco > 15].index[-1]
        CORRIGIDAS.setdefault(f"{nome}: {int((buraco > 15).sum())} buracos de mais de 15 dias; usa só a partir de {ini:%d/%m/%Y}", None)
        d = d.loc[ini:]
    return d


def atr(d, n=20):
    h, l, c = d["High"], d["Low"], d["Close"]
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def rsi(c, n):
    dif = c.diff()
    return 100 - 100 / (1 + dif.clip(lower=0).ewm(alpha=1 / n, adjust=False).mean()
                        / (-dif.clip(upper=0)).ewm(alpha=1 / n, adjust=False).mean())


BASE = {t: limpa(dados[t], t) for t in TICKERS}
MERC = {t: ("B3" if t.endswith(".SA") else "EUA") for t in TICKERS}
IND = {"B3": ibov["Close"], "EUA": spx["Close"]}

## Parte 1: redundância entre indicadores

In [ ]:
def indicadores(d, indice):
    c, h, l, v = d["Close"], d["High"], d["Low"], d["Volume"]
    ema = lambda s, n: s.ewm(span=n, adjust=False).mean()
    macd = ema(c, 12) - ema(c, 26)
    trix = ema(ema(ema(c, 15), 15), 15).pct_change()
    k = 100 * (c - l.rolling(14).min()) / (h.rolling(14).max() - l.rolling(14).min())
    ix = indice.reindex(c.index).ffill()
    return pd.DataFrame({
        "IFR(14)": rsi(c, 14),
        "Estocástico %K(14)": k.rolling(3).mean(),
        "MACD (histograma)": (macd - ema(macd, 9)) / c,
        "Trix(15)": trix,
        "ROC(20)": c.pct_change(20),
        "Força relativa (63 dias)": c.pct_change(63) - ix.pct_change(63),
        "Volume relativo (20 dias)": v / v.rolling(20).mean(),
        "Distância da MM200": c / c.rolling(200).mean() - 1,
    }).dropna()


mats = [indicadores(BASE[t], IND[MERC[t]]).corr(method="spearman") for t in TICKERS]
CORR = pd.concat(mats).groupby(level=0).median().loc[mats[0].index, mats[0].columns]
mostra("Parte 1. Correlação mediana (Spearman, 40 ações) entre indicadores")
mostra(CORR.round(2).to_string())
mom = ["IFR(14)", "Estocástico %K(14)", "MACD (histograma)", "Trix(15)", "ROC(20)"]
dentro = CORR.loc[mom, mom].to_numpy()[np.triu_indices(len(mom), 1)]
mostra(f"   entre os 5 osciladores de momento: correlação mediana {np.median(dentro):.2f};"
      f" do IFR com o volume relativo: {CORR.loc['IFR(14)', 'Volume relativo (20 dias)']:.2f}")

# ---------------- preparação comum às partes 2 a 5 ----------------
PREP = {}
for m in ("B3", "EUA"):
    acima = {}
    for t in [t for t in TICKERS if MERC[t] == m]:
        c = BASE[t]["Close"]
        acima[t] = (c > c.rolling(200).mean()).where(c.rolling(200).mean().notna())
    amp = pd.DataFrame(acima).mean(axis=1)                       # % das 20 ações acima da própria MM200
    ix = IND[m]
    fr = pd.DataFrame({t: BASE[t]["Close"].pct_change(126) for t in acima})
    fr = fr.sub(ix.reindex(fr.index).ffill().pct_change(126), axis=0)
    rank = fr.rank(axis=1, pct=True)                             # força relativa de 6 meses, posição no grupo
    ctx = ((ix > ix.rolling(200).mean()).reindex(amp.index).ffill().fillna(False) & (amp > 0.5))
    for t in acima:
        d = BASE[t]
        c = d["Close"]
        mm200 = c.rolling(200).mean()
        PREP[t] = pd.DataFrame({
            "contexto": ctx.reindex(d.index).fillna(False),
            "classificacao": (rank[t].reindex(d.index) >= 0.5),
            "ciclo": (c > mm200) & (mm200 > mm200.shift(20)),
            "gatilho": c > d["High"].rolling(20).max().shift(1),
            "nivel": d["High"].rolling(20).max().shift(1),
            "atr": atr(d),
        })
    PREP[m + "_amp"] = amp


def operar(t, dias, entrada="fechamento"):
    """Uma posição por vez: entra no fechamento do dia do sinal (ou na ordem stop no nível, durante o pregão),
    stop de 3 ATR, sai no fechamento 20 pregões depois. Resultado em R, com custo."""
    d, p = BASE[t], PREP[t]
    O, H, L, C = (d[x].to_numpy() for x in ("Open", "High", "Low", "Close"))
    A, NIV = p["atr"].to_numpy(), p["nivel"].to_numpy()
    out, livre = [], 0
    for i in dias:
        if i < 260 or i > len(C) - PRAZO - 2 or i < livre:
            continue
        if entrada == "fechamento":
            e, risco = C[i], 3 * A[i]
        else:                                                    # ordem stop na máxima de 20 dias, durante o pregão
            e, risco = max(NIV[i], O[i]), 3 * A[i - 1]
        stop, saida = e - risco, None
        if entrada != "fechamento" and C[i] <= stop:          # no próprio dia da entrada, só se sabe o fechamento
            saida = C[i]
        for k in range(i + 1, i + PRAZO + 1):
            if saida is not None:
                break
            if L[k] <= stop:
                saida = min(stop, O[k])
        saida = C[i + PRAZO] if saida is None else saida
        out.append({"ativo": t, "data": d.index[i], "R": (saida - e) / risco - CUSTO * e / risco,
                    "falso_no_dia": C[i] < NIV[i]})
        livre = i + PRAZO
    return out


def resumo(df):
    r = df["R"].to_numpy()
    return f"{len(r):_} operações | acerto {np.mean(r > 0):.0%} | expectativa {r.mean():+.3f} R"

## Parte 2: a bendita confirmação

In [ ]:
fech, intra = [], []
for t in TICKERS:
    p, d = PREP[t], BASE[t]
    fech += operar(t, np.flatnonzero(p["gatilho"].to_numpy()), "fechamento")
    tocou = (d["High"] > p["nivel"]).to_numpy()
    intra += operar(t, np.flatnonzero(tocou), "intradiario")
F, I = pd.DataFrame(fech), pd.DataFrame(intra)
mostra("\nParte 2. Rompimento da máxima de 20 dias, stop de 3 ATR, saída em 20 pregões (40 ações, de 2006 a 2026)")
mostra("   esperando o fechamento acima do nível:   ", resumo(F))
mostra("   ordem stop no nível, durante o pregão:   ", resumo(I))
mostra(f"   nos rompimentos do pregão, {I['falso_no_dia'].mean():.0%} fecharam o dia abaixo do nível;"
      f" esses deram {I.loc[I['falso_no_dia'], 'R'].mean():+.3f} R e os confirmados {I.loc[~I['falso_no_dia'], 'R'].mean():+.3f} R")

## Parte 3: as camadas do 6C

In [ ]:
CAMADAS = [("Candle: só o gatilho (rompimento de 20 dias)", ["gatilho"]),
           ("+ Ciclo: acima da MM200 subindo", ["gatilho", "ciclo"]),
           ("+ Classificação: força relativa na metade de cima", ["gatilho", "ciclo", "classificacao"]),
           ("+ Contexto: índice acima da MM200 e mais da metade das ações também", ["gatilho", "ciclo", "classificacao", "contexto"])]
TAB = []
for nome, filtros in CAMADAS:
    ops, nulos = [], []
    for t in TICKERS:
        p = PREP[t]
        mask = np.logical_and.reduce([p[f].to_numpy(dtype=bool) for f in filtros])
        ops += operar(t, np.flatnonzero(mask))
    df = pd.DataFrame(ops)
    # hipótese nula: dias sorteados que passam pelos mesmos filtros, sem o gatilho (mesma quantidade de operações por ativo)
    filtros_sem = [f for f in filtros if f != "gatilho"]
    nul_medias = []
    elegiveis = {}
    for t in TICKERS:
        p = PREP[t]
        mk = np.logical_and.reduce([p[f].to_numpy(dtype=bool) for f in filtros_sem]) if filtros_sem else np.ones(len(p), bool)
        elegiveis[t] = np.flatnonzero(mk)
    cont = df["ativo"].value_counts()
    for _ in range(200):
        r = []
        for t, n in cont.items():
            ok = elegiveis[t]
            if len(ok) == 0:
                continue
            sorteio = np.sort(rng.choice(ok, size=min(len(ok), n * 3), replace=False))
            r += [o["R"] for o in operar(t, sorteio)[:n]]
        nul_medias.append(np.mean(r))
    nul_medias = np.array(nul_medias)
    dentro_ = df[df["data"] < CORTE]["R"]
    fora_ = df[df["data"] >= CORTE]["R"]
    TAB.append({"camada": nome, "operações": len(df), "acerto": np.mean(df["R"] > 0), "expectativa_R": df["R"].mean(),
                "de 2006 a 2015": dentro_.mean(), "de 2016 a 2026": fora_.mean(), "n_fora": len(fora_),
                "acaso_R": nul_medias.mean(), "acaso_p95": np.percentile(nul_medias, 95),
                "vence_o_acaso": np.mean(nul_medias < df["R"].mean())})
T = pd.DataFrame(TAB).set_index("camada")
mostra("\nParte 3. As camadas do 6C (mesmo gatilho, mesmo stop, mesma saída; acaso = 200 sorteios de dias com os mesmos filtros)")
mostra(T.round(3).to_string())

## Parte 4: duas estratégias juntas

In [ ]:
def ops_ifr2(t):
    d = BASE[t]
    c = d["Close"]
    C = c.to_numpy()
    sinal = ((c > c.rolling(200).mean()) & (rsi(c, 2) < 10)).to_numpy()
    sai = (c > c.rolling(5).mean()).to_numpy()
    out, livre = [], 0
    for i in np.flatnonzero(sinal):
        if i < 260 or i > len(C) - 12 or i < livre:
            continue
        k = next((k for k in range(i + 1, i + 11) if sai[k]), i + 10)
        out.append({"data": d.index[k], "r": C[k] / C[i] - 1 - CUSTO})
        livre = k + 1
    return out


def ops_romp(t):
    d, p = BASE[t], PREP[t]
    C = d["Close"].to_numpy()
    out, livre = [], 0
    for i in np.flatnonzero(p["gatilho"].to_numpy() & p["ciclo"].to_numpy()):
        if i < 260 or i > len(C) - PRAZO - 2 or i < livre:
            continue
        out.append({"data": d.index[i + PRAZO], "r": C[i + PRAZO] / C[i] - 1 - CUSTO})
        livre = i + PRAZO
    return out


mens = {}
for nome, fn in (("Rompimento", ops_romp), ("IFR(2)", ops_ifr2)):
    o = pd.DataFrame([x for t in TICKERS for x in fn(t)])
    mens[nome] = o.groupby(o["data"].dt.to_period("M"))["r"].mean() * 100   # resultado médio por operação no mês (%)
M = pd.DataFrame(mens).fillna(0).loc["2006":]
M["Metade de cada"] = M.mean(axis=1)
mostra("\nParte 4. Resultado médio por operação, mês a mês (%), de 2006 a 2026")
mostra(f"   correlação entre as duas: {M['Rompimento'].corr(M['IFR(2)']):.2f}")
for col in M:
    s = M[col]
    mostra(f"   {col:15s} média {s.mean():+.2f} | desvio {s.std():.2f} | média/desvio {s.mean() / s.std():.2f}"
          f" | pior mês {s.min():+.1f} | meses negativos {np.mean(s < 0):.0%}")

## Parte 5: o painel 6C em 30/09/2026

In [ ]:
hist = pd.DataFrame([o for t in TICKERS for o in operar(t, np.flatnonzero(PREP[t]["gatilho"].to_numpy()))])
linhas = []
for t in [t for t in TICKERS if MERC[t] == "B3"]:
    p, d = PREP[t], BASE[t]
    u = p.index[-1]
    h_t = hist[hist["ativo"] == t]["R"]
    ultimo_g = p.index[p["gatilho"].to_numpy()].max()
    linhas.append({"ativo": t.replace(".SA", ""), "preço": d["Close"].iloc[-1],
                   "contexto": "ok" if p["contexto"].iloc[-1] else "não",
                   "FR 6m (posição)": f"{(p['classificacao'].iloc[-1] and 'metade de cima') or 'metade de baixo'}",
                   "ciclo": "alta" if p["ciclo"].iloc[-1] else "não",
                   "gatilho hoje": "sim" if p["gatilho"].iloc[-1] else "não",
                   "último gatilho": f"{ultimo_g:%d/%m/%Y}" if pd.notna(ultimo_g) else "-",
                   "stop 3 ATR": d["Close"].iloc[-1] - 3 * p["atr"].iloc[-1],
                   "ações p/ R$ 200 de risco": int(200 // (3 * p["atr"].iloc[-1])),
                   "histórico do gatilho (R)": f"{h_t.mean():+.2f} em {len(h_t)}"})
PAINEL = pd.DataFrame(linhas).set_index("ativo")
amp = PREP["B3_amp"].iloc[-1]
mostra(f"\nParte 5. Painel 6C em {PREP[TICKERS[0]].index[-1]:%d/%m/%Y} (B3). Ibovespa acima da MM200: "
      f"{bool(ibov['Close'].iloc[-1] > ibov['Close'].rolling(200).mean().iloc[-1])}; das 20 ações, {amp:.0%} acima da MM200")
mostra(PAINEL.round(2).to_string())
mostra(f"   nível do rompimento de ITUB4 no dia: R$ {PREP['ITUB4.SA']['nivel'].iloc[-1]:.2f}")
mostra(f"   o que a limpeza corrigiu nas 40 ações ({len(CORRIGIDAS)} casos):")
for linha_ in CORRIGIDAS:
    mostra("      " + linha_)

In [ ]:
# Painel 6C para a sua lista: troque os códigos e rode de novo as Partes 1 e 5 (com download novo, apague o cache).
PAINEL.sort_values("FR 6m (posição)")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, a função `operar` da Parte 1. Mostre a diferença entre a entrada no fechamento e a entrada `intradiario`: por que a entrada no pregão usa `max(NIV[i], O[i])` e `3 * A[i - 1]`, e por que, no dia da entrada, só se confere o fechamento." Confira com a Parte 2, que imprime +0,148 R esperando o fechamento, +0,155 R entrando no pregão e 47% de rompimentos do pregão que fecharam o dia abaixo do nível.
2. **Adaptar:** "Acrescente [ATIVO], uma ação da B3, à lista `B3` da célula de abertura, apague o cache e rode de novo as Partes 1 e 5. Não mude os filtros, o stop de 3 ATR nem a perda máxima de R$ 200." Antes de mudar a lista, confira se a Parte 5 mostra ITUB4 a R$ 44,26, com stop de 3 ATR em R$ 41,23 e 65 ações, como no livro.

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 1, acrescente o Williams %R e veja com quem ele se parece.
2. Na Parte 2, troque o rompimento de 20 por 55 dias. Esperar o fechamento passa a valer mais?
3. Na Parte 3, troque o filtro de contexto (amplitude acima de 50%) por amplitude abaixo de 50%. O que acontece com a expectativa?
4. Na Parte 5, monte o painel para 20 ações americanas.

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.